# Module 4 — Monte Carlo Simulation

**Why Monte Carlo?**

A single-point projection ('your portfolio will be worth ₹X in 20 years') is intellectually dishonest — it implies certainty that doesn't exist. Monte Carlo replaces it with a **distribution of outcomes**.

We simulate 10,000 possible futures using **block bootstrapping** from real historical returns. This preserves:
- **Fat tails** — crash years are sampled from real crash years, not smoothed away
- **Serial correlation** — good years tend to cluster, bad years too (momentum)

Output: 'You have an **X% chance** of reaching your goal' — the headline number that makes this tool real.

In [ ]:
# ── Setup (Colab) ────────────────────────────────────────────────
!pip install -q numpy pandas PyPortfolioOpt plotly yfinance
import os
if not os.path.exists('robo-advisor'):
    !git clone https://github.com/parinmody30/robo-advisor.git
else:
    !git -C robo-advisor pull

import sys
sys.path.insert(0, 'robo-advisor/src')
DATA_DIR = 'robo-advisor/data'
print('Setup complete.')

In [ ]:
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from data_loader import load_all, SYNTHETIC_ASSETS
from monte_carlo import run_simulation, run_all_goals

# Load saved outputs from previous modules
with open(f'{DATA_DIR}/risk_profile.json') as f:
    profile = json.load(f)
with open(f'{DATA_DIR}/allocation.json') as f:
    allocation = json.load(f)
with open(f'{DATA_DIR}/goals.json') as f:
    goals_data = json.load(f)

persona        = profile['persona']
weights        = allocation['weights']
monthly_savings = goals_data['monthly_savings']
goals          = goals_data['goals']

print(f'Persona: {persona} | Monthly savings: ₹{monthly_savings:,}')
print(f'Goals: {[g["name"] for g in goals]}')

## 4A. Fetch market data
*(Needed for historical return bootstrapping)*

In [ ]:
prices, mu, cov = load_all(start='2015-01-01')
print('Data loaded. Years of history:', round(len(prices)/252, 1))

## 4B. Run Monte Carlo for primary goal
Deep dive on the highest-priority goal first.

In [ ]:
primary = goals[0]
print(f'Running 10,000 simulations for: {primary["name"]}')
print(f'Target corpus: ₹{primary["future_value"]:,.0f} in {primary["years_to_goal"]} years')

mc = run_simulation(
    prices=prices,
    weights=weights,
    synthetic_assets=SYNTHETIC_ASSETS,
    initial_investment=0,
    monthly_sip=monthly_savings,
    horizon_years=primary['years_to_goal'],
    target_corpus=primary['future_value'],
    n_sims=10_000,
)

print(f'\n✅ Probability of success : {mc.prob_success}%')
print(f'⚡ After 2008-style crash  : {mc.stress_prob_success}%')
print(f'📊 Median terminal value  : ₹{mc.percentiles["p50"].iloc[-1]:,.0f}')

## 4C. Probability cone — the money chart

In [ ]:
years = mc.percentiles.index
p     = mc.percentiles

fig = go.Figure()

# Shaded bands
fig.add_trace(go.Scatter(
    x=list(years)+list(years[::-1]),
    y=list(p['p90'])+list(p['p10'][::-1]),
    fill='toself', fillcolor='rgba(21,101,192,0.10)',
    line=dict(color='rgba(0,0,0,0)'), name='10th–90th percentile',
))
fig.add_trace(go.Scatter(
    x=list(years)+list(years[::-1]),
    y=list(p['p75'])+list(p['p25'][::-1]),
    fill='toself', fillcolor='rgba(21,101,192,0.20)',
    line=dict(color='rgba(0,0,0,0)'), name='25th–75th percentile',
))

# Median line
fig.add_trace(go.Scatter(
    x=years, y=p['p50'], mode='lines',
    name='Median outcome',
    line=dict(color='#1565C0', width=3),
))

# Target line
fig.add_hline(
    y=primary['future_value'], line_dash='dash', line_color='#E53935',
    annotation_text=f'Target: ₹{primary["future_value"]/1e5:.0f}L',
    annotation_position='top left',
)

fig.update_layout(
    title=f'{primary["name"]} — Monte Carlo Projection ({mc.prob_success}% success)',
    xaxis_title='Years', yaxis_title='Portfolio Value (₹)',
    template='plotly_white', height=500,
)
fig.show()

## 4D. Stress test — 2008-style crash in Year 1

In [ ]:
sp = mc.stress_percentiles

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=years, y=p['p50'], mode='lines',
    name='Base case (median)', line=dict(color='#1565C0', width=2.5),
))
fig2.add_trace(go.Scatter(
    x=years, y=sp['p50'], mode='lines',
    name='After -45% crash in Yr 1 (median)', line=dict(color='#E53935', width=2.5, dash='dot'),
))
fig2.add_hline(
    y=primary['future_value'], line_dash='dash', line_color='#388E3C',
    annotation_text='Target', annotation_position='top left',
)
fig2.update_layout(
    title=f'Stress Test: Base vs 2008-style Crash | Success: {mc.prob_success}% → {mc.stress_prob_success}%',
    xaxis_title='Years', yaxis_title='Portfolio Value (₹)',
    template='plotly_white', height=460,
)
fig2.show()
print(f'\nSuccess probability drops from {mc.prob_success}% → {mc.stress_prob_success}% after crash.')
print(f'Recovery visible in the median path — this is why long horizons matter.')

## 4E. All goals — success probability table

In [ ]:
print('Running simulations for all goals...')
all_results = run_all_goals(
    prices=prices,
    weights=weights,
    synthetic_assets=SYNTHETIC_ASSETS,
    goals=goals,
    monthly_savings=monthly_savings,
    n_sims=10_000,
)

print(f'\n{"Goal":<25} {"Horizon":>8} {"SIP/mo":>10} {"Success":>10} {"After Crash":>12}')
print('-' * 68)
for r in all_results:
    flag = '✅' if r['prob_success'] >= 70 else '⚠️ ' if r['prob_success'] >= 50 else '❌'
    print(f'{flag} {r["goal"]:<23} {r["horizon"]:>6}yr  ₹{r["monthly_sip"]:>8,.0f}  {r["prob_success"]:>8.1f}%  {r["stress_prob"]:>10.1f}%')

## 4F. Terminal value distribution — histogram

In [ ]:
fig3 = go.Figure()
fig3.add_trace(go.Histogram(
    x=mc.terminal_values / 1e5,
    nbinsx=80, marker_color='#1565C0', opacity=0.75,
    name='Terminal wealth distribution',
))
fig3.add_vline(
    x=primary['future_value']/1e5, line_dash='dash', line_color='#E53935',
    annotation_text='Target', annotation_position='top right',
)
fig3.update_layout(
    title=f'Distribution of Terminal Wealth — {primary["name"]} ({primary["years_to_goal"]}yr horizon)',
    xaxis_title='Terminal Value (₹ Lakhs)',
    yaxis_title='Number of simulations',
    template='plotly_white', height=420,
)
fig3.show()
print(f'Area to the right of the red line = {mc.prob_success}% probability of success')

## Theory note

**Why block bootstrap instead of parametric Monte Carlo?**

The standard parametric approach draws returns from a normal distribution fitted to historical data. This underestimates tail risk because:
1. Real returns have **excess kurtosis** (fat tails) — crashes happen 5-10x more often than a normal distribution predicts
2. Returns have **autocorrelation** — bull markets and bear markets cluster

Block bootstrapping samples consecutive 3-year blocks from actual history, so real crash sequences (2008, 2020) appear in our simulations at their actual frequency and magnitude.

**Interpreting the cone:**
- The **median (p50)** is your most likely outcome — not a guarantee
- The **p10 band** represents a persistently bad scenario — still planning for this is prudent
- **Probability of success** = fraction of 10,000 paths where terminal wealth ≥ target corpus